# Rising Waters: Flood Prediction using Machine Learning

**Project Overview:**  
This notebook builds a flood prediction pipeline using four classification algorithms — Decision Tree, Random Forest, KNN, and XGBoost. The best-performing model is saved for integration with the Flask web application.

---

**Dataset:** Flood Prediction Dataset (Kaggle — arbejthi/rainfall-dataset)  
**Target Variable:** `FLOOD` (1 = Flood, 0 = No Flood)  
**Features Used:** Annual Rainfall, Cloud Visibility, Seasonal Rainfall (Jun–Sep), Temperature, Humidity

## Epic 1 — Importing Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

try:
    from xgboost import XGBClassifier
    USE_XGBOOST_LIB = True
    print('XGBoost library found — using XGBClassifier.')
except ImportError:
    USE_XGBOOST_LIB = False
    print('XGBoost library not found — using sklearn GradientBoostingClassifier as substitute.')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully.')

## Epic 2 — Reading and Exploring the Dataset

In [ ]:
# Load the dataset — place 'flood dataset.xlsx' inside the ../dataset/ folder
DATASET_PATH = os.path.join('..', 'dataset', 'flood dataset.xlsx')

df = pd.read_excel(DATASET_PATH)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print('Columns:', list(df.columns))

In [ ]:
df.head(10)

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
df.describe()

### Column Configuration

Update the cell below if your dataset uses different column names.  
Run `print(df.columns.tolist())` (cell above) to see your exact column names.

In [ ]:
# ── Column names confirmed from actual dataset ───────────────────────────────
TARGET_COL   = 'flood'         # Target: 1 = Flood, 0 = No Flood
COL_ANNUAL   = 'ANNUAL'        # Annual rainfall (mm)
COL_CLOUD    = 'Cloud Cover'   # Cloud visibility / cover (%)
COL_SEASONAL = 'Jun-Sep'       # Seasonal rainfall Jun–Sep (mm)
COL_TEMP     = 'Temp'          # Temperature (°C)
COL_HUMIDITY = 'Humidity'      # Humidity (%)

# Feature column order — MUST match app.py prediction input order
FEATURE_COLS = [COL_ANNUAL, COL_CLOUD, COL_SEASONAL, COL_TEMP, COL_HUMIDITY]
print('Feature columns:', FEATURE_COLS)
print('Target column:', TARGET_COL)

# Verify columns exist
missing = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
if missing:
    print(f'\nWARNING: These columns were not found: {missing}')
    print('Available columns:', df.columns.tolist())
else:
    print('\nAll columns verified successfully.')

## Epic 2 — Univariate Analysis

Distribution plots and box plots for each numeric feature to understand individual variable behaviour.

In [ ]:
numeric_cols = [c for c in FEATURE_COLS if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

fig, axes = plt.subplots(2, len(numeric_cols), figsize=(4 * len(numeric_cols), 8), squeeze=False)
fig.suptitle('Univariate Analysis — Distribution & Box Plots', fontsize=14, fontweight='bold')

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[0, i], color='steelblue')
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_xlabel('')

    sns.boxplot(y=df[col], ax=axes[1, i], color='lightcoral')
    axes[1, i].set_title(f'{col} — Box Plot')

plt.tight_layout()
plt.show()

## Epic 2 — Multivariate Analysis

A correlation heatmap reveals relationships between features and helps identify redundant or highly correlated variables.

In [ ]:
cols_for_corr = [c for c in FEATURE_COLS + [TARGET_COL] if c in df.columns]
corr = df[cols_for_corr].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, annot_kws={'size': 11})
plt.title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
if TARGET_COL in df.columns:
    plt.figure(figsize=(5, 4))
    ax = sns.countplot(x=df[TARGET_COL], palette=['steelblue', 'tomato'])
    ax.set_xticklabels(['No Flood (0)', 'Flood (1)'])
    plt.title('Target Variable Distribution', fontweight='bold')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    print(df[TARGET_COL].value_counts())

## Epic 3 — Data Pre-Processing

### Step 1: Handle Missing Values

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

# Fill numeric missing values with median, categorical with mode
for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

print('\nAfter handling — Missing values:')
print(df.isnull().sum())

### Step 2: Handle Outliers (IQR Capping)

In [ ]:
def cap_outliers(dataframe, columns):
    df_capped = dataframe.copy()
    for col in columns:
        if col not in df_capped.columns:
            continue
        Q1 = df_capped[col].quantile(0.25)
        Q3 = df_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = ((df_capped[col] < lower) | (df_capped[col] > upper)).sum()
        df_capped[col] = df_capped[col].clip(lower, upper)
        if outliers:
            print(f'{col}: {outliers} outlier(s) capped (lower={lower:.2f}, upper={upper:.2f})')
    return df_capped

numeric_features = [c for c in FEATURE_COLS if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
df = cap_outliers(df, numeric_features)
print('\nOutlier capping complete.')

### Step 3: Handle Categorical Variables

In [ ]:
le = LabelEncoder()
categorical_cols = [c for c in df.columns if df[c].dtype == 'object']

if categorical_cols:
    print('Encoding categorical columns:', categorical_cols)
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    print('Encoding complete.')
else:
    print('No categorical columns found — no encoding required.')

df.head()

### Step 4: Separate Features (X) and Target (y)

In [ ]:
available_features = [c for c in FEATURE_COLS if c in df.columns]
X = df[available_features]
y = df[TARGET_COL]

print('Feature matrix X shape:', X.shape)
print('Target vector y shape:', y.shape)
print('Feature columns (in order):', list(X.columns))
print('\nClass distribution:')
print(y.value_counts())

### Step 5: Train–Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

### Step 6: Feature Scaling (StandardScaler)

Formula: z = (x − μ) / σ  
The scaler is **fitted on training data only** and then applied to the test set to prevent data leakage.

In [ ]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled  = sc.transform(X_test)

print('Feature scaling applied.')
print('Scaler mean:', sc.mean_.round(3))
print('Scaler std: ', sc.scale_.round(3))

## Epic 4 — Model Building

Four classification models are trained and evaluated using the same metrics:  
**Accuracy Score**, **Confusion Matrix**, and **Classification Report**.

### Model 1: Decision Tree

In [ ]:
def decisiontree(X_train, X_test, y_train, y_test):
    clf = DecisionTreeClassifier(random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print('──── Decision Tree ────────────────────────')
    print(f'Accuracy: {acc * 100:.2f}%')
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Flood', 'Flood']))
    return clf, y_pred, acc

dt_model, dt_pred, dt_acc = decisiontree(X_train_scaled, X_test_scaled, y_train, y_test)

### Model 2: Random Forest

In [ ]:
def randomForest(X_train, X_test, y_train, y_test):
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print('──── Random Forest ────────────────────────')
    print(f'Accuracy: {acc * 100:.2f}%')
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Flood', 'Flood']))
    return clf, y_pred, acc

rf_model, rf_pred, rf_acc = randomForest(X_train_scaled, X_test_scaled, y_train, y_test)

### Model 3: K-Nearest Neighbors (KNN)

In [ ]:
def KNN(X_train, X_test, y_train, y_test):
    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print('──── K-Nearest Neighbors (k=5) ────────────')
    print(f'Accuracy: {acc * 100:.2f}%')
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Flood', 'Flood']))
    return clf, y_pred, acc

knn_model, knn_pred, knn_acc = KNN(X_train_scaled, X_test_scaled, y_train, y_test)

### Model 4: XGBoost

In [ ]:
def xgboost_model(X_train, X_test, y_train, y_test):
    if USE_XGBOOST_LIB:
        clf = XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        )
    else:
        clf = GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=42
        )

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    label = 'XGBoost' if USE_XGBOOST_LIB else 'GradientBoosting'
    print(f'──── {label} ────────────────────────────')
    print(f'Accuracy: {acc * 100:.2f}%')
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('\nClassification Report:')
    print(classification_report(y_test, y_pred, target_names=['No Flood', 'Flood']))
    return clf, y_pred, acc

xgb_model, xgb_pred, xgb_acc = xgboost_model(X_train_scaled, X_test_scaled, y_train, y_test)

### Comparing All Models

In [ ]:
def compareModel(accuracies):
    names = list(accuracies.keys())
    scores = [v * 100 for v in accuracies.values()]

    print('╔══════════════════════════════════════════════╗')
    print('║          MODEL PERFORMANCE COMPARISON        ║')
    print('╠══════════════════════════════════════════════╣')
    for name, score in zip(names, scores):
        bar = '█' * int(score / 5)
        print(f'║  {name:<22} {score:>6.2f}%  {bar}')
    print('╚══════════════════════════════════════════════╝')

    # Bar chart
    plt.figure(figsize=(8, 4))
    colors = ['steelblue', 'seagreen', 'darkorange', 'crimson']
    bars = plt.bar(names, scores, color=colors, edgecolor='white', linewidth=0.8)
    plt.ylim(0, 110)
    plt.ylabel('Accuracy (%)')
    plt.title('Model Comparison — Test Accuracy', fontweight='bold')
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 f'{score:.2f}%', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

    best = max(accuracies, key=accuracies.get)
    print(f'\nBest model: {best} with {accuracies[best] * 100:.2f}% accuracy.')

model_accuracies = {
    'Decision Tree':  dt_acc,
    'Random Forest':  rf_acc,
    'KNN':            knn_acc,
    'XGBoost':        xgb_acc
}

compareModel(model_accuracies)

## Saving the Best Model and Scaler

XGBoost is selected as the final model due to its superior generalization on structured data.  
Both the trained model and the fitted scaler must be saved so the Flask app can make predictions.

In [ ]:
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_SAVE_PATH  = os.path.join(MODELS_DIR, 'floods.save')
SCALER_SAVE_PATH = os.path.join(MODELS_DIR, 'transform.save')

joblib.dump(xgb_model, MODEL_SAVE_PATH)
joblib.dump(sc, SCALER_SAVE_PATH)

print(f'Model saved  → {MODEL_SAVE_PATH}')
print(f'Scaler saved → {SCALER_SAVE_PATH}')
print(f'\nFinal XGBoost accuracy on test data: {xgb_acc * 100:.2f}%')
print(f'Feature order for prediction: {list(X.columns)}')
print('\nDeployment ready. Start the Flask app with: python app.py')

---

## Summary

| Step | Action |
|------|--------|
| Data Loading | `pd.read_excel()` — loaded flood dataset |
| EDA | head, shape, info, describe, distribution plots, heatmap |
| Preprocessing | Missing value imputation, IQR capping, label encoding |
| Splitting | 80/20 stratified train-test split |
| Scaling | StandardScaler (z = (x-μ)/σ) applied to features |
| Models | Decision Tree, Random Forest, KNN, XGBoost |
| Best Model | XGBoost — saved as `floods.save` |
| Scaler | Saved as `transform.save` for consistent inference |

> **Next:** Run `python app.py` in the project root to launch the Flask web application.